# pcap → baseline vs qaccess_t throughput (tshark + CSV)

**Goal**: compare **baseline/off** vs **qaccess_t** from the same eval session.

- Session: `logs_exp/session_qaccess_t_20260524_220428`
- Runs: `fig7_baseline` and `fig7_qaccess_t`
- Method (same as single-run notebook): `tshark -e frame.time_relative -e frame.len`, 1 s bins, filter `udp` only.
- **Total = pathA_all_udp + pathB_all_udp** per second.

Fig7 BW steps (server egress): 0–50s @ 20 Mbps → 50–100s @ 30 Mbps → 100s+ @ 10 Mbps.


In [10]:
# Third-party deps (network only on first install)
import importlib.util
import subprocess
import sys

_PACKAGES = ["pandas", "matplotlib"]

def _have(mod: str) -> bool:
    return importlib.util.find_spec(mod) is not None

_missing = [p for p in _PACKAGES if not _have(p)]
if _missing:
    print("Installing:", ", ".join(_missing), flush=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", *_missing],
        timeout=600,
    )
else:
    print("OK:", ", ".join(_PACKAGES))

OK: pandas, matplotlib


In [11]:
import math
import os
import shutil
import subprocess
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython import get_ipython

    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except (ImportError, AttributeError):
    os.environ.setdefault("MPLBACKEND", "Agg")


def find_repo() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "scripts" / "analyze" / "parse_logs.py").is_file():
            return p
    return cwd


REPO = find_repo()
print("REPO =", REPO)

REPO = /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment


In [ ]:
# --- Q-ACCeSS-T eval session (TIMEOUT=420, SAVE_LOGS=0) ---
SESSION = REPO / "logs_exp" / "session_qaccess_t_20260524_220428"

RUNS = {
    "baseline": SESSION / "fig7_baseline",
    "qaccess_t": SESSION / "fig7_qaccess_t",
}

OUT_DIR = SESSION / "compare_csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILTER_ALL_UDP = "udp"
SECOND_MAX = 420
POST_STEADY_SECOND = 100
PATH_B_ALERT_MBPS = 10.5

FIG_TOTAL = OUT_DIR / "baseline_vs_qaccess_t_total_1s.png"
FIG_PATHS = OUT_DIR / "baseline_vs_qaccess_t_paths_1s.png"


def find_pcaps(run_dir):
    pcaps = sorted((run_dir / "pcaps").glob("path*.pcap"))
    pcap_a = next(p for p in pcaps if "pathA" in p.name)
    pcap_b = next(p for p in pcaps if "pathB" in p.name)
    return pcap_a, pcap_b


for label, run_dir in RUNS.items():
    pcap_a, pcap_b = find_pcaps(run_dir)
    print(f"[{label}]")
    print(" ", pcap_a, "exists" if pcap_a.is_file() else "MISSING")
    print(" ", pcap_b, "exists" if pcap_b.is_file() else "MISSING")

assert shutil.which("tshark"), "tshark not on PATH (install Wireshark)"
print("OUT_DIR =", OUT_DIR)


In [ ]:

def read_frame_len_bins(pcap, display_filter):
    """Bin total frame bytes by second index: floor(frame.time_relative)."""
    bins = defaultdict(int)
    cmd = [
        "tshark", "-r", str(pcap), "-Y", display_filter,
        "-T", "fields", "-E", "separator=\t",
        "-e", "frame.time_relative", "-e", "frame.len",
    ]
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or f"tshark failed: {pcap}")
    for line in proc.stdout.splitlines():
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        try:
            si = int(math.floor(float(parts[0])))
            flen = int(parts[1])
        except ValueError:
            continue
        if si >= 0:
            bins[si] += flen
    return bins


def bins_to_mbps_series(bin_dict, last_second, interval_s):
    return pd.Series(
        [bin_dict.get(s, 0) * 8 / 1_000_000 / interval_s for s in range(0, last_second + 1)],
        index=range(0, last_second + 1),
        dtype="float64",
    )


def load_run_throughput_1s(label, run_dir):
    pcap_a, pcap_b = find_pcaps(run_dir)
    print(f"Reading [{label}] pcaps (all UDP)...", flush=True)
    bins_a = read_frame_len_bins(pcap_a, FILTER_ALL_UDP)
    bins_b = read_frame_len_bins(pcap_b, FILTER_ALL_UDP)
    last_s = int(max(SECOND_MAX, max(bins_a.keys(), default=0), max(bins_b.keys(), default=0)))
    df = pd.DataFrame({
        "second": range(0, last_s + 1),
        "pathA_all_udp_mbps": bins_to_mbps_series(bins_a, last_s, 1.0).values,
        "pathB_all_udp_mbps": bins_to_mbps_series(bins_b, last_s, 1.0).values,
    })
    df["total_all_udp_mbps"] = df["pathA_all_udp_mbps"] + df["pathB_all_udp_mbps"]
    csv_path = OUT_DIR / f"throughput_all_udp_1s_{label}.csv"
    df.to_csv(csv_path, index=False)
    print(f"  Wrote {csv_path} rows={len(df)}")
    return df, bins_a, bins_b, last_s


run_data = {}
for label, run_dir in RUNS.items():
    df, ba, bb, last_s = load_run_throughput_1s(label, run_dir)
    run_data[label] = {"df": df, "bins_a": ba, "bins_b": bb, "last_s": last_s}

df_baseline = run_data["baseline"]["df"]
df_qaccess_t = run_data["qaccess_t"]["df"]
last_s = max(run_data["baseline"]["last_s"], run_data["qaccess_t"]["last_s"])
print("done. last_s =", last_s)


In [ ]:
# Mean Mbps summary: baseline vs qaccess_t

WINDOWS = [
    ("0-50", 0, 50),
    ("50-100", 50, 100),
    ("100-420", 100, 421),
    ("full", 0, last_s + 1),
]


def mean_in_window(df, lo, hi, col="total_all_udp_mbps"):
    w = df[(df["second"] >= lo) & (df["second"] < hi)]
    return float(w[col].mean()) if len(w) else float("nan")


rows = []
for wname, lo, hi in WINDOWS:
    b = mean_in_window(df_baseline, lo, hi)
    t = mean_in_window(df_qaccess_t, lo, hi)
    imp = (t - b) / b * 100.0 if b > 0 else float("nan")
    rows.append({
        "window": wname,
        "baseline_total_mbps": round(b, 3),
        "qaccess_t_total_mbps": round(t, 3),
        "improvement_pct": round(imp, 2),
    })

df_compare = pd.DataFrame(rows)
compare_csv = OUT_DIR / "baseline_vs_qaccess_t_windows.csv"
df_compare.to_csv(compare_csv, index=False)
display(df_compare)
print("Wrote", compare_csv)

_KEYS = ("File size", "Number of packets", "Capture duration", "Data size", "Data bit rate")
if shutil.which("capinfos"):
    for label, run_dir in RUNS.items():
        pcap_a, pcap_b = find_pcaps(run_dir)
        for path_label, pcap in [("Path A", pcap_a), ("Path B", pcap_b)]:
            print(f"\n[{label}] {path_label}: {pcap.name}")
            out = subprocess.run(["capinfos", str(pcap)], capture_output=True, text=True)
            for line in out.stdout.splitlines():
                if any(k in line for k in _KEYS):
                    print(" ", line.strip())


In [ ]:
cols = ["second", "pathA_all_udp_mbps", "pathB_all_udp_mbps", "total_all_udp_mbps"]
show = (
    df_baseline["second"].between(40, 60)
    | df_baseline["second"].between(90, 130)
)
print("baseline sample (40-60s, 90-130s)")
display(df_baseline.loc[show, cols].head(20).reset_index(drop=True))
print("qaccess_t sample (40-60s, 90-130s)")
display(df_qaccess_t.loc[show, cols].head(20).reset_index(drop=True))


In [ ]:
# Path B after 100s (10 Mbps cap window)
for label, df in [("baseline", df_baseline), ("qaccess_t", df_qaccess_t)]:
    post = df[df["second"] >= POST_STEADY_SECOND]
    violations = post[post["pathB_all_udp_mbps"] > PATH_B_ALERT_MBPS]
    print(f"[{label}] seconds pathB > {PATH_B_ALERT_MBPS} after t>={POST_STEADY_SECOND}: {len(violations)}")
    if len(post):
        print(f"  max_pathB ≈ {post['pathB_all_udp_mbps'].max():.4f} Mbps")
        print(f"  avg_pathB ≈ {post['pathB_all_udp_mbps'].mean():.4f} Mbps")
        print(f"  avg_total ≈ {post['total_all_udp_mbps'].mean():.4f} Mbps")


In [ ]:
width = 10


def bytes_in_seconds(bin_dict, start_s, end_s):
    return sum(bin_dict.get(s, 0) for s in range(start_s, end_s))


def to_10s_df(bins_a, bins_b):
    rows10 = []
    for start in range(0, SECOND_MAX + 1, width):
        end = start + width
        ba = bytes_in_seconds(bins_a, start, end)
        bb = bytes_in_seconds(bins_b, start, end)
        a_mbps = ba * 8 / 1_000_000 / width
        b_mbps = bb * 8 / 1_000_000 / width
        rows10.append((start, end, a_mbps, b_mbps, a_mbps + b_mbps))
    return pd.DataFrame(
        rows10,
        columns=["time_start_s", "time_end_s", "pathA_all_udp_mbps", "pathB_all_udp_mbps", "total_all_udp_mbps"],
    )


for label in RUNS:
    d10 = to_10s_df(run_data[label]["bins_a"], run_data[label]["bins_b"])
    p = OUT_DIR / f"throughput_all_udp_10s_{label}.csv"
    d10.to_csv(p, index=False)
    print(f"Wrote {p}")
    display(d10.head(6))


In [ ]:
# --- Main comparison: total throughput (baseline vs qaccess_t) ---
plt.figure(figsize=(12, 5))
plt.plot(df_baseline["second"], df_baseline["total_all_udp_mbps"], label="baseline total", linewidth=2)
plt.plot(df_qaccess_t["second"], df_qaccess_t["total_all_udp_mbps"], label="qaccess_t total", linewidth=2)
plt.axvline(50, linestyle="--", linewidth=1, color="gray", label="BW step 50s")
plt.axvline(100, linestyle="--", linewidth=1, color="gray", label="BW step 100s")
plt.xlabel("Time from pcap start (s)")
plt.ylabel("Throughput (Mbps)")
plt.title("Fig7 eval: baseline vs qaccess_t — total all-UDP (1 s bins)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_TOTAL, dpi=200)
plt.show()
print("saved", FIG_TOTAL)


In [ ]:
# --- Path A / Path B comparison (same style as single-run notebook) ---
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df_baseline["second"], df_baseline["pathA_all_udp_mbps"], label="baseline Path A")
axes[0].plot(df_qaccess_t["second"], df_qaccess_t["pathA_all_udp_mbps"], label="qaccess_t Path A")
axes[0].set_ylabel("Mbps")
axes[0].set_title("Path A (all UDP)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_baseline["second"], df_baseline["pathB_all_udp_mbps"], label="baseline Path B")
axes[1].plot(df_qaccess_t["second"], df_qaccess_t["pathB_all_udp_mbps"], label="qaccess_t Path B")
axes[1].set_ylabel("Mbps")
axes[1].set_xlabel("Time from pcap start (s)")
axes[1].set_title("Path B (all UDP)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.axvline(50, linestyle="--", linewidth=1, color="gray")
    ax.axvline(100, linestyle="--", linewidth=1, color="gray")

fig.suptitle("Fig7 eval: baseline vs qaccess_t — per-path throughput (1 s bins)", y=1.02)
plt.tight_layout()
plt.savefig(FIG_PATHS, dpi=200, bbox_inches="tight")
plt.show()
print("saved", FIG_PATHS)
